### Geographic Information Extraction Script
This script uses the city names from `Cities.csv` to fetch coordinates (latitude, longitude) and altitude.
The results are formatted as a compressed Python dictionary `CITY_GEO_DICT = {...}` which can be directly copied into your preprocessing notebook.

In [ ]:
!pip install geopy geocoder requests -q
import pandas as pd
import geocoder
from geopy.geocoders import Nominatim
import requests
import time
import json

def get_altitude(lat, lon):
    if lat is None or lon is None:
        return None
    try:
        # Use Open-Meteo Elevation API (Free, no key required)
        url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            return round(response.json().get('elevation', [0.0])[0], 1)
    except:
        pass
    return 0.0

def generate_geo_dictionary():
    df_cities = pd.read_csv("../provided/Cities.csv")
    
    geolocator = Nominatim(user_agent="mm_geo_fetcher_2026")
    city_dict = {}
    
    print(f"Starting to cache {len(df_cities)} cities (Estimated time: ~10 minutes)...")
    
    for idx, row in df_cities.iterrows():
        city_id = row['CityID']
        city_name = row['City']
        state_abb = row['State']
        
        # Resolve country based on State abbreviations if necessary
        country = "USA"
        if str(state_abb).strip().upper() == "MX":
            country = "Mexico"
        elif str(state_abb).strip().upper() == "PR":
            country = "Puerto Rico"
            
        query = f"{city_name}, {state_abb}, {country}"
        lat, lon = None, None
        
        try:
            # 1st Attempt: Nominatim (Most accurate, but fails on some cities)
            location = geolocator.geocode(query, timeout=5)
            if location:
                lat, lon = location.latitude, location.longitude
            else:
                # 2nd Attempt: ArcGIS geocoder (Covers Canada, Ireland, and small towns better)
                g = geocoder.arcgis(query)
                if g.ok:
                    lat, lon = g.latlng
            
            if lat is not None and lon is not None:
                alt = get_altitude(lat, lon)
                city_dict[int(city_id)] = {"Lat": round(lat, 4), "Lon": round(lon, 4), "Alt": alt}
                print(f"[{idx+1}/{len(df_cities)}] Success: {query} -> {lat:.2f}, {lon:.2f}, {alt}m")
            else:
                city_dict[int(city_id)] = {"Lat": None, "Lon": None, "Alt": None}
                print(f"[{idx+1}/{len(df_cities)}] Failed: {query}")
            
            time.sleep(1) # Sleep to prevent API rate limiting
            
        except Exception as e:
            print(f"[{idx+1}/{len(df_cities)}] Connection Error ({query}): {e}")
            city_dict[int(city_id)] = {"Lat": None, "Lon": None, "Alt": None}
            time.sleep(2) # Longer sleep on error
            
    return city_dict

# Execute dictionary generation
city_geo_dict = generate_geo_dictionary()

# Compress the dictionary output to one line per city
dict_str_lines = ["{"]
for cid, coord in city_geo_dict.items():
    coord_str = json.dumps(coord)
    dict_str_lines.append(f"    {cid}: {coord_str},")
dict_str_lines.append("}")

final_dict_str = "\n".join(dict_str_lines)

# Print the results to the screen (Copy & Paste this part)
print("\n" + "="*50)
print("Copy the code below and paste it into a new cell (`CITY_GEO_DICT = ...`) in your preprocessing notebook:")
print("="*50 + "\n")
print(f"CITY_GEO_DICT = {final_dict_str}")

Starting to cache 509 cities (Estimated time: ~10 minutes)...
[1/509] Success: Abilene, TX, USA -> 32.45, -99.75, 531.0m
[2/509] Success: Akron, OH, USA -> 41.08, -81.52, 292.0m
[3/509] Success: Albany, NY, USA -> 42.65, -73.75, 38.0m
[4/509] Success: Albuquerque, NM, USA -> 35.08, -106.65, 1514.0m
[5/509] Success: Allentown, PA, USA -> 40.60, -75.47, 112.0m
[6/509] Success: Ames, IA, USA -> 42.03, -93.62, 285.0m
[7/509] Success: Amherst, MA, USA -> 42.37, -72.52, 100.0m
[8/509] Success: Anaheim, CA, USA -> 33.83, -117.91, 54.0m
[9/509] Success: Anchorage, AK, USA -> 61.22, -149.89, 39.0m
[10/509] Success: Ann Arbor, MI, USA -> 42.28, -83.75, 255.0m
[11/509] Success: Annapolis, MD, USA -> 38.98, -76.49, 19.0m
[12/509] Success: Arlington, TX, USA -> 32.74, -97.11, 191.0m
[13/509] Success: Asheville, NC, USA -> 35.60, -82.55, 674.0m
[14/509] Success: Athens, GA, USA -> 33.96, -83.38, 236.0m
[15/509] Success: Athens, OH, USA -> 39.36, -82.06, 252.0m
[16/509] Success: Atlanta, GA, USA -> 3